# Chapter 10 — A Revolver, Not a Foundation

**Companion to Applied AI**

Question: Can one model win a chamber and lose the next?

By the end of this notebook you will have:

- configured logical chambers instead of hard-coding model IDs
- compared candidates per chamber on pass/fail, cost, and flips
- computed negative flips and cost per passing item

## What this notebook demonstrates
A synthetic per-chamber comparison. Candidates, costs, and pass/fail grids are invented to expose the *mechanism* (chamber-dependent winners, negative flips).

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

seed: 42


## 1. Chambers are configuration, not model names

In [2]:
chambers = {
    "fast-classify": {"tasks": 60, "weight": 1},
    "draft":         {"tasks": 40, "weight": 2},
    "deep-review":   {"tasks": 25, "weight": 3},
    "critic":        {"tasks": 25, "weight": 2},
}
candidates = {
    "candidate-X": {"cost_per_task": 0.002, "latency_s": 1.2},
    "incumbent-Y": {"cost_per_task": 0.010, "latency_s": 4.0},
}
print("chambers:", list(chambers), "\ncandidates:", list(candidates))

chambers: ['fast-classify', 'draft', 'deep-review', 'critic'] 
candidates: ['candidate-X', 'incumbent-Y']


## 2. Per-chamber outcomes on a frozen task set

In [3]:
import random as _r
# pass probabilities per (candidate, chamber): X wins some chambers, loses others
P = {("candidate-X", "fast-classify"): 0.90, ("incumbent-Y", "fast-classify"): 0.82,
     ("candidate-X", "draft"):         0.70, ("incumbent-Y", "draft"):         0.78,
     ("candidate-X", "deep-review"):   0.60, ("incumbent-Y", "deep-review"):   0.80,
     ("candidate-X", "critic"):        0.75, ("incumbent-Y", "critic"):        0.72}
results = {}
for ch, cfg in chambers.items():
    for cand in candidates:
        rng = _r.Random(hash((SEED, ch, cand)) % (2**32))
        passes = [1 if rng.random() < P[(cand, ch)] else 0 for _ in range(cfg["tasks"])]
        results[(cand, ch)] = passes
for ch in chambers:
    for cand in candidates:
        r = results[(cand, ch)]
        print(f"{cand:14s} {ch:14s} pass rate {sum(r)/len(r):.2f}")

candidate-X    fast-classify  pass rate 0.87
incumbent-Y    fast-classify  pass rate 0.82
candidate-X    draft          pass rate 0.70
incumbent-Y    draft          pass rate 0.80
candidate-X    deep-review    pass rate 0.72
incumbent-Y    deep-review    pass rate 0.84
candidate-X    critic         pass rate 0.72
incumbent-Y    critic         pass rate 0.60


## 3. Negative flips + cost per passing item decide chamber by chamber

In [4]:
for ch, cfg in chambers.items():
    x, y = results[("candidate-X", ch)], results[("incumbent-Y", ch)]
    neg_flips = sum(1 for a, b in zip(y, x) if a == 1 and b == 0)  # incumbent passed, candidate fails
    gain = sum(x) / len(x) - sum(y) / len(y)
    cpp = candidates["candidate-X"]["cost_per_task"] / (sum(x) / len(x))
    print(f"{ch:14s} mean gain {gain:+.2f}  negative flips {neg_flips:2d}/{len(x)}  X cost/passing ${cpp:.4f}")
    assert 0 <= neg_flips <= len(x)

fast-classify  mean gain +0.05  negative flips  7/60  X cost/passing $0.0023
draft          mean gain -0.10  negative flips 11/40  X cost/passing $0.0029
deep-review    mean gain -0.12  negative flips  5/25  X cost/passing $0.0028
critic         mean gain +0.12  negative flips  2/25  X cost/passing $0.0028


## Interpretation
- Supports: a single global winner need not exist; release is a per-chamber decision trading gain against flips and cost.
- Does NOT support: any claim about real models X or Y.

## Try it yourself
1. Change a chamber's task count and watch cost/passing move.
2. Price a negative flip (e.g. escaped defect = $50) and recompute the winner.
3. Add a third candidate that only contests `deep-review`.